# Неделя 1: цены Binance и разведка русскоязычных сайтов

Запускайте ячейки сверху вниз кнопкой слева от каждой. Ячейка выполнена, когда вместо круга появляется галочка.

**Важно:** перед запуском включите GPU не нужно — на этом шаге хватит обычного режима.

## 1. Подключаем Google Drive

Данные будут лежать на вашем Диске, а не в Colab: Colab стирает свои файлы через несколько часов, Диск — нет.

Появится окно с запросом доступа — разрешите.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Указываем, где код и где данные

В строке `REPO_URL` замените `<ваш-логин>` на ваше имя пользователя на GitHub.

In [ ]:
import os

REPO_URL = 'https://github.com/<ваш-логин>/crypto-news-forecasting.git'
DATA_DIR = '/content/drive/MyDrive/cryptonews-data'

os.makedirs(DATA_DIR, exist_ok=True)
os.environ['CRYPTONEWS_DATA_DIR'] = DATA_DIR
print('Данные будут здесь:', DATA_DIR)

## 3. Скачиваем код и ставим его

Если ячейка ругается `repository not found`, проверьте имя пользователя в `REPO_URL` и что репозиторий публичный.

In [ ]:
!rm -rf /content/repo
!git clone -q $REPO_URL /content/repo
%cd /content/repo
!pip install -q -r requirements.txt
!pip install -q -e .
print('Готово')

## 4. Проверяем окружение

Скрипт выпишет версии библиотек и запишет их в файл. Это ответ рецензенту на вопрос о версиях (замечание 1). Ошибок быть не должно.

In [ ]:
!python scripts/00_check_env.py

## 5. Качаем цены

Около 220 помесячных архивов по двум активам, примерно 5–15 минут. У каждого файла сверяется контрольная сумма.

Если связь оборвётся — просто запустите ячейку заново: скачанное берётся из кэша на Диске.

In [ ]:
!python scripts/01_download_prices.py

## 6. Разведка русскоязычных сайтов

Несколько десятков запросов с паузой, примерно 2–3 минуты. Скрипт выясняет, можно ли вообще собирать новости с каждого сайта.

**Весь вывод этой ячейки пришлите мне** — по нему решим, какие сайты берём.

In [ ]:
!python scripts/01a_probe_ru_sites.py

## 7. Что получилось

Список созданных файлов и первые строки таблицы цен.

In [ ]:
import pandas as pd, os
for root, _, files in os.walk(DATA_DIR):
    for f in sorted(files):
        if not f.endswith('.zip'):
            p = os.path.join(root, f)
            print(f'{os.path.getsize(p)/1e6:8.1f} МБ  {p}')
print()
btc = pd.read_parquet(f'{DATA_DIR}/raw/binance/BTCUSDT_1h.parquet')
print('BTCUSDT:', btc.index[0], '…', btc.index[-1], '| строк:', len(btc))
display(btc.head())